# R1 Rejector: Best Validation Accuracy

Boxplot der besten Validierungsgenauigkeit pro Run fuer die vier Rejector-Architekturen im `embedding`-Ordner.

In [ ]:
from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

mpl.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": True,
    "legend.framealpha": 0.95,
})

In [ ]:
RUN_ROOT = Path(
    "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/"
    "tensorboard_runs/tensorboard_runs_rejector_r2/embedding"
).resolve()

PLOTS_DIR = Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/plot").resolve()
OUT_DIR = PLOTS_DIR / "figures_r2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ARCHITECTURES = {
    "conv-mlp": "conv_mlp",
    "resnet-small": "resnet_small",
}

RUN_ROOT, OUT_DIR

In [ ]:
def read_accuracy_val(run_dir: Path) -> list[dict]:
    scalar_dir = run_dir / "Accuracy_val"
    event_files = sorted(scalar_dir.glob("events.out.tfevents.*"))
    if not event_files:
        return []

    accumulator = EventAccumulator(str(scalar_dir), size_guidance={"scalars": 0})
    accumulator.Reload()
    tags = accumulator.Tags().get("scalars", [])
    if not tags:
        return []

    tag = "Accuracy_val" if "Accuracy_val" in tags else tags[0]
    return [
        {"epoch": event.step, "wall_time": event.wall_time, "value": float(event.value), "tb_tag": tag}
        for event in accumulator.Scalars(tag)
    ]


def load_best_validation_accuracies(run_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    best_rows = []
    run_rows = []

    for architecture, dirname in ARCHITECTURES.items():
        architecture_dir = run_root / dirname
        run_dirs = sorted(p for p in architecture_dir.iterdir() if p.is_dir() and p.name.startswith("run_"))

        for run_dir in run_dirs:
            meta = {
                "architecture": architecture,
                "architecture_dir": dirname,
                "run": run_dir.name,
                "run_path": str(run_dir),
            }

            try:
                events = read_accuracy_val(run_dir)
            except Exception as exc:
                warnings.warn(f"Konnte {run_dir / 'Accuracy_val'} nicht lesen: {exc}")
                events = []

            run_rows.append({
                **meta,
                "n_val_points": len(events),
                "status": "ok" if events else "kein Accuracy_val",
            })

            if not events:
                continue

            best_event = max(events, key=lambda event: event["value"])
            best_rows.append({
                **meta,
                "best_val_accuracy": best_event["value"],
                "best_epoch": best_event["epoch"],
                "n_val_points": len(events),
            })

    best = pd.DataFrame(best_rows)
    runs = pd.DataFrame(run_rows)
    return best, runs

In [ ]:
best_val, runs = load_best_validation_accuracies(RUN_ROOT)

summary = (
    runs.groupby("architecture", sort=False)
    .agg(
        runs=("run", "size"),
        runs_with_accuracy_val=("n_val_points", lambda values: int((values > 0).sum())),
    )
    .reindex(ARCHITECTURES.keys())
)

print(f"Geladen: {len(best_val)} Runs mit Accuracy_val aus {len(runs)} Run-Verzeichnissen")
display(summary)
display(best_val.sort_values(["architecture", "best_val_accuracy"], ascending=[True, False]).head(10))

In [ ]:
PALETTE = {
    "conv-mlp": "#1f77b4",
    "resnet-small": "#ff7f0e",
}

labels = list(ARCHITECTURES.keys())
box_data = [
    best_val.loc[best_val["architecture"] == label, "best_val_accuracy"].dropna().to_numpy()
    for label in labels
]

fig, ax = plt.subplots(figsize=(8.6, 5.2), constrained_layout=True)
box = ax.boxplot(
    box_data,
    labels=labels,
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "linewidth": 1.5},
    whiskerprops={"color": "0.35", "linewidth": 1.2},
    capprops={"color": "0.35", "linewidth": 1.2},
)

for patch, label in zip(box["boxes"], labels):
    patch.set_facecolor(PALETTE[label])
    patch.set_alpha(0.22)
    patch.set_edgecolor(PALETTE[label])
    patch.set_linewidth(1.4)

rng = np.random.default_rng(42)
for index, label in enumerate(labels, start=1):
    values = best_val.loc[best_val["architecture"] == label, "best_val_accuracy"].dropna().to_numpy()
    jitter = rng.normal(0, 0.045, size=len(values))
    ax.scatter(
        np.full(len(values), index) + jitter,
        values,
        s=28,
        color=PALETTE[label],
        edgecolor="white",
        linewidth=0.45,
        alpha=0.82,
        zorder=3,
    )

ax.set_xlabel("Architektur")
ax.set_ylabel("Beste Validierungsgenauigkeit")
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1.0, decimals=0))
ax.grid(True, axis="y", linestyle=":", alpha=0.7, color="gray")
ax.grid(False, axis="x")
ax.tick_params(axis="x", rotation=15)

y_values = best_val["best_val_accuracy"].dropna()
if not y_values.empty:
    ypad = max((float(y_values.max()) - float(y_values.min())) * 0.08, 0.01)
    ax.set_ylim(max(0.0, float(y_values.min()) - ypad), min(1.0, float(y_values.max()) + ypad))

png_path = OUT_DIR / "r2_rejector_best_val_accuracy_boxplot.png"
pdf_path = OUT_DIR / "r2_rejector_best_val_accuracy_boxplot.pdf"
fig.savefig(png_path)
fig.savefig(pdf_path)

png_path, pdf_path